 ## Assignment 1 — Python Fundamentals for Data Streaming

### Import libraries

In [1]:
import json
import pandas as pd
from shuttle import parse_event
from shuttle import validate_event
from shuttle import event_stream
from shuttle import get_occupancy

## Part A
### 1.Load the dataset
The shuttle dataset is loaded from the JSON sample. JSON objects are read into Python dictionaries, producing a list of dictionaries where each dictionary represents one shuttle event.

In [2]:
with open("events.json", "r") as file:
    events = json.load(file)

print(type(events))
print(type(events[0]))
print("number of events:", len(events))
events[0]

<class 'list'>
<class 'dict'>
number of events: 10


{'timestamp': '08:00',
 'route': 'aitu-campus-residence',
 'bus': 'b01',
 'passengers': 18,
 'speed_kmh': 31,
 'status': 'on_route'}

### 2. Parsing

In [23]:
event1 = events[0]
event1

{'timestamp': '08:00',
 'route': 'aitu-campus-residence',
 'bus': 'b01',
 'passengers': 18,
 'speed_kmh': 31,
 'status': 'on_route'}

In [22]:
print(type(event1["timestamp"]))
print(type(event1["passengers"]))
print(type(event1["speed_kmh"]))
print(type(event1["status"]))

<class 'str'>
<class 'int'>
<class 'int'>
<class 'str'>


In [5]:
event1 = parse_event(events[0])
event1

{'timestamp': datetime.time(8, 0),
 'route': 'aitu-campus-residence',
 'bus': 'b01',
 'passengers': 18,
 'speed_kmh': 31,
 'status': 'on_route'}

In [6]:
print(type(event1["timestamp"]))
print(type(event1["passengers"]))
print(type(event1["speed_kmh"]))
print(type(event1["status"]))

<class 'datetime.time'>
<class 'int'>
<class 'int'>
<class 'str'>


### 3. Event validation

In [7]:
check = validate_event(events[0])
check

{'valid': True,
 'event': {'timestamp': datetime.time(8, 0),
  'route': 'aitu-campus-residence',
  'bus': 'b01',
  'passengers': 18,
  'speed_kmh': 31,
  'status': 'on_route'},
 'errors': []}

In [8]:
bad_event = {
    "timestamp": "08:38",
    "route": "aitu-campus-residence",
    "bus": "b70",
    "passengers": -5,
    "speed_kmh": 145,
    "status": "flying"
}
validate_event(bad_event)

{'valid': False,
 'event': {'timestamp': datetime.time(8, 38),
  'route': 'aitu-campus-residence',
  'bus': 'b70',
  'passengers': -5,
  'speed_kmh': 145,
  'status': 'flying'},
 'errors': ['passengers cannot be negative',
  'speed must be between 0 and 120',
  'status must be on_route or stopped']}

### 4. Generator

In [9]:
stream = event_stream(events)
print(next(stream))
print(next(stream))
print(next(stream))

{'timestamp': '08:00', 'route': 'aitu-campus-residence', 'bus': 'b01', 'passengers': 18, 'speed_kmh': 31, 'status': 'on_route'}
{'timestamp': '08:01', 'route': 'aitu-campus-residence', 'bus': 'b02', 'passengers': 22, 'speed_kmh': 28, 'status': 'on_route'}
{'timestamp': '08:02', 'route': 'aitu-campus-residence', 'bus': 'b01', 'passengers': 21, 'speed_kmh': 29, 'status': 'on_route'}


## Part B. Streaming simulation

In [ ]:
stream = event_stream(events)
results = []

for event_number in range(1, 6):
    raw_event = next(stream)
    check = validate_event(raw_event)

    if check["valid"]:
        event = check["event"]
        reason = "valid event"

        results.append({
            "event": event_number,
            "passengers": event["passengers"],
            "speed": event["speed_kmh"],
            "status": event["status"],
            "processed": True,
            "reason": reason
        })

    else:
        results.append({
            "event": event_number,
            "passengers": raw_event.get("passengers"),
            "speed": raw_event.get("speed_kmh"),
            "status": raw_event.get("status"),
            "processed": False,
            "reason": ", ".join(check["errors"])
        })

In [11]:
result_table = pd.DataFrame(results)
result_table

,event,passengers,speed,status,processed,reason
0,1,18,31,on_route,True,valid event
1,2,22,28,on_route,True,valid event
2,3,21,29,on_route,True,valid event
3,4,25,27,on_route,True,valid event
4,5,24,0,stopped,True,valid event


## Part C. JSON and REST API Processing 

In [12]:
for passengers in [0, 10, 11, 20, 21, 30, 31]:
    print(passengers, get_occupancy(passengers))

0 low
10 low
11 medium
20 medium
21 high
30 high
31 over_capacity


## Part D. Analysis

In [13]:
parsed_events = []
for raw_event in event_stream(events):
    check = validate_event(raw_event)

    if check["valid"]:
        parsed_events.append(check["event"])

data = pd.DataFrame(parsed_events)
data

,timestamp,route,bus,passengers,speed_kmh,status
0,08:00:00,aitu-campus-residence,b01,18,31,on_route
1,08:01:00,aitu-campus-residence,b02,22,28,on_route
2,08:02:00,aitu-campus-residence,b01,21,29,on_route
3,08:03:00,aitu-campus-residence,b02,25,27,on_route
4,08:04:00,aitu-campus-residence,b01,24,0,stopped
5,08:05:00,aitu-campus-residence,b02,26,30,on_route
6,08:06:00,aitu-campus-residence,b01,23,32,on_route
7,08:07:00,aitu-campus-residence,b02,28,26,on_route
8,08:08:00,aitu-campus-residence,b01,27,25,on_route
9,08:09:00,aitu-campus-residence,b02,30,24,on_route


In [14]:
avg_passengers = data["passengers"].mean()
avg_passengers

np.float64(24.4)

In [15]:
max_passengers = data["passengers"].max()
max_passengers

30

In [16]:
stopped = (data["status"] == "stopped").sum()
print(stopped)

1


In [17]:
busiest_minute = data.loc[data["passengers"].idxmax()]
busiest_minute

timestamp                  08:09:00
route         aitu-campus-residence
bus                             b02
passengers                       30
speed_kmh                        24
status                     on_route
Name: 9, dtype: object

In [18]:
bus_totals = (data.groupby("bus")["passengers"].sum())
bus_totals

bus
b01    113
b02    131
Name: passengers, dtype: int64

In [19]:
busiest_bus = bus_totals.idxmax()
busiest_bus

'b02'

In [20]:
print(type(events))
print(type(events[0]))
print("number of events:", len(events))
events[0]

<class 'list'>
<class 'dict'>
number of events: 10


{'timestamp': '08:00',
 'route': 'aitu-campus-residence',
 'bus': 'b01',
 'passengers': 18,
 'speed_kmh': 31,
 'status': 'on_route'}